# Lab 0: Basic Python Programming and Data Workflow

## Overview

In this lab, you will review basic Python programming skills and use them in a simple public health data workflow, from reading and inspecting a CSV file to preparing data and fitting a basic regression model.

## Learning goals

By the end of this lab, you should be able to:

- use basic Python tools such as lists, dictionaries, loops, conditionals, and functions;
- read a CSV file into pandas and inspect the structure of a dataset;
- prepare a small working dataset from a larger public health dataset;
- identify and handle missing values;
- create a derived variable for analysis;
- implement and evaluate a basic train/test regression workflow.

## Before you begin: using Google Colab

Google Colab is a browser-based notebook environment for Python. It lets you write text and code in the same document and run code one cell at a time.

A few basic controls are enough for this lab:
- Click inside a code cell and press **Shift + Enter** to run it.
- If a cell fails, read the error message carefully. It often tells you exactly what went wrong.
- Run cells from top to bottom. If you skip a setup cell, later cells may fail because an object has not been created yet.
- You do not need to install anything for this lab beyond the standard Colab environment.

## 1. Basic Python review

This section reviews a few Python ideas that you will use repeatedly in data analysis.

### Lists and dictionaries

A **list** stores an ordered collection of items.
A **dictionary** stores information using named keys.

In data analysis, a list is often used to store variable names.
A dictionary is useful when we want to represent one record, profile, or set of labeled values.

In [ ]:
risk_factors = ["smoking", "obesity", "inactivity", "uninsured"]

county_profile = {
    "county": "River County",
    "smoking": 18.4,
    "obesity": 31.2,
    "inactivity": 24.1
}

print(risk_factors)
print(county_profile)
print(county_profile["county"])

In the example above, `risk_factors` is a list of variable names.
The object `county_profile` is a dictionary describing one county.

Notice the difference in access:

- for a list, we usually access by position;
- for a dictionary, we access by key.

In [ ]:
print(risk_factors[0])
print(county_profile["obesity"])

### Accessing values and looping through variables

A common task in public health data analysis is to repeat the same action across several variables.
A `for` loop helps us do that efficiently.

In [ ]:
obesity_rate = county_profile["obesity"]

for factor in risk_factors:
    print("Risk factor:", factor)

### Conditional statements

Conditionals let us apply simple rules.
For example, we can compare a value to a threshold and print a message based on the result.

In [ ]:
if obesity_rate >= 30:
    print("This county has a high obesity rate.")
else:
    print("This county does not have a high obesity rate.")

### Writing a simple function

A function lets us reuse logic without rewriting the same code many times.
This is especially useful when the same rule needs to be applied to many observations or variables.

In [ ]:
def classify_risk(value, threshold):
    if value >= threshold:
        return "high"
    return "lower"

print(classify_risk(18.4, 15))
print(classify_risk(10.2, 15))

### Quick practice

In the next cell, change the threshold from `15` to `20`.
Then run the cell again and see whether the classification changes.

In [ ]:
print(classify_risk(18.4, 15))
print(classify_risk(10.2, 15))

## 2. Reading a real CSV file

We now move from small Python objects to a real public health dataset.


Before working with a real dataset, we need to import a few Python packages.

A **package** is a collection of tools that adds useful functionality to Python. In data analysis, we often import packages instead of writing everything from scratch.

In this lab, we will use two core packages:

- **pandas** helps us read, store, inspect, and manipulate tabular data such as CSV files;
- **NumPy** provides numerical tools that support many common operations in data analysis.

The `import` statement tells Python to load these packages so that we can use their functions in the notebook. We also give them short names, `pd` and `np`, which are standard abbreviations in Python data analysis.

In [ ]:
import pandas as pd
import numpy as np

### About the dataset

In this lab, we will use data from **the National Health and Nutrition Examination Survey (NHANES)**. NHANES has collected health and nutrition data over many years. Our lab dataset is based on the 2009–2012 survey waves and includes information on about 10,000 U.S. residents. It contains a broad set of demographic, physical, nutritional, and lifestyle variables.

#### Reference

CDC NHANES official website: https://https://wwwn.cdc.gov/nchs/nhanes/


### Read the full NHANES file

The command `pd.read_csv()` reads a CSV file into a pandas DataFrame.
A DataFrame is the main table-like object used in pandas.

Here we pass a **URL** instead of a local file path.
That means pandas will download the CSV behind the scenes and read it directly into Python.

After reading the file, we inspect the first few rows before making any changes.


In [ ]:
# private repo version
nhanes = pd.read_csv("https://raw.githubusercontent.com/statOmics/PSLSData/main/NHANES.csv")
nhanes.head()


### Inspect the size and structure of the dataset

When you receive a new dataset, it is good practice to ask a few basic questions:

- How many rows and columns does it have?
- What are the variable names?
- Which variables are numeric, and which are categorical?

These checks help you avoid mistakes later.

In [ ]:
print("Shape:", nhanes.shape)
print("\nFirst 15 columns:")
print(nhanes.columns[:15].tolist())

In [ ]:
nhanes.info()

### Descriptive summary

The next command gives a quick statistical summary for numeric variables.
This is a fast way to detect unusual ranges and get a first impression of the data.

In [ ]:
nhanes.describe().T.head(10)

## 3. Choosing a working subset of variables

The full file contains many variables, but we do not need all of them for one small exercise.
Selecting a smaller working subset is a normal part of data preparation.

Here we imagine a simple public health question:

**Can a few demographic and health-related variables help us predict average systolic blood pressure?**


### Step 1: store selected variable names in a list

Notice that we are using a Python list again.

In [ ]:
selected_columns = [
    "Age",
    "Gender",
    "Race1",
    "Education",
    "BMI",
    "Weight",
    "Height",
    "Pulse",
    "TotChol",
    "BPSysAve",
    "Diabetes",
    "PhysActive"
]

selected_columns

### Step 2: create a smaller working DataFrame

This step keeps only the variables we want for the current lab.
We are not deleting the original dataset.
We are simply creating a smaller version for focused work.

In [ ]:
df = nhanes[selected_columns].copy()
df.head()

### Compare the full file and the working subset


In [ ]:
print("Full NHANES shape:", nhanes.shape)
print("Working subset shape:", df.shape)

## 4. Identifying missing values

Missing data is common in public health research.
Some people may skip a question, miss a laboratory measurement, or not complete a physical exam.
For that reason, missing-value checks should happen early in the workflow.

The next command counts missing values in each variable.

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts

### Missing-value proportions

Raw counts are useful, but proportions can be easier to interpret.
For example, 50 missing values may be small in a large dataset and large in a small dataset.

In [ ]:
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False).round(1)
missing_percent

### Brief interpretation

Look at the output above and identify:

- one variable with no missing values,
- one variable with a moderate amount of missingness,
- one variable with relatively high missingness.


## 5. Simple cleaning decisions

There is no single correct way to handle missing data.
The right approach depends on the research question, the amount of missingness, and the modeling strategy.

For an introductory lab, we will keep the workflow simple:

1. keep a manageable set of variables,
2. create one derived variable,
3. remove rows with missing values in the variables needed for a small regression model.

This is not the most advanced approach.
It is just a clear starting point.


### Create a derived variable

Derived variables are variables created from existing information.
They are common in applied work.

Below, we create a log-transformed version of total cholesterol.
A log transformation is often used for positive continuous variables when we want a more compact scale and a variable that may be easier to model.

We will call this new variable `LogTotChol`.


In [ ]:
df["LogTotChol"] = np.log(df["TotChol"])
df[["TotChol", "LogTotChol"]].head()


### Inspect the new variable

The table below compares the original variable and its log-transformed version.


In [ ]:
df[["TotChol", "LogTotChol"]].describe().T


### Prepare a smaller modeling dataset

For the first modeling exercise, we will use only numeric variables.

Our outcome will be:

- `BPSysAve`: average systolic blood pressure

Our predictors will be:

- `Age`
- `BMI`
- `Weight`
- `Height`
- `Pulse`
- `LogTotChol`

Notice that we now use the derived variable `LogTotChol` in the model instead of the original `TotChol`.


In [ ]:
model_columns = ["Age", "BMI", "Weight", "Height", "Pulse", "LogTotChol", "BPSysAve"]
model_df = df[model_columns].copy()
model_df.head()


### Remove rows with missing values in the modeling variables

For this introductory lab, we will use complete cases for the selected modeling variables.
That means we keep only rows with no missing values in this smaller modeling dataset.

This is a simple teaching choice, not a universal recommendation.

In [ ]:
print("Shape before dropping missing values:", model_df.shape)

model_df = model_df.dropna()

print("Shape after dropping missing values:", model_df.shape)

## 6. Quick exploration

Before fitting a model, it is often useful to look at a few summaries and plots.
This can help you detect obvious patterns and possible issues.



In [ ]:
model_df.describe().T

### Correlations among numeric variables

This is a quick way to see which variables appear related.
Correlation does not imply causation, but it can be useful for early exploration.

In [ ]:
model_df.corr(numeric_only=True).round(2)

### Simple plots

We will make two very basic plots:

- a histogram of `LogTotChol`;
- a scatter plot of `LogTotChol` versus systolic blood pressure.


#### Importing a plotting package

We will use **Matplotlib** to create simple visualizations in this lab. Matplotlib is one of the most common plotting packages in Python.

The line below imports its plotting functions and gives them the short name `plt`, which is a standard abbreviation.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.hist(model_df["LogTotChol"], bins=25, edgecolor="black")
plt.xlabel("Log total cholesterol")
plt.ylabel("Count")
plt.title("Distribution of log total cholesterol")
plt.show()


In [ ]:
plt.scatter(model_df["LogTotChol"], model_df["BPSysAve"], alpha=0.4)
plt.xlabel("Log total cholesterol")
plt.ylabel("Average systolic blood pressure")
plt.title("Log total cholesterol and systolic blood pressure")
plt.show()


## 7. A basic train/test workflow

We now move to a very simple predictive workflow.

The purpose here is not to build the best model.
The purpose is to practice the standard sequence:

1. define predictors and outcome,
2. split the data into training and test sets,
3. fit a model on the training set,
4. evaluate it on the test set.

### A note on scikit-learn

To fit and evaluate a simple model, we will use **scikit-learn**, one of the most widely used Python libraries for machine learning.

In this lab, we will use three parts of scikit-learn:

- `train_test_split`, which helps us divide the data into a training set and a test set;
- `LinearRegression`, which fits a basic linear regression model;
- `mean_squared_error` and `r2_score`, which help us evaluate model performance.

These tools allow us to carry out a simple train/test workflow in a standard way.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

### Define predictors and outcome


In [ ]:
X = model_df[["Age", "BMI", "Weight", "Height", "Pulse", "LogTotChol"]]
y = model_df["BPSysAve"]

print("X shape:", X.shape)
print("y shape:", y.shape)


### Split the data into training and test sets

We will place 80% of observations in the training set and 20% in the test set.
The argument `random_state=2026` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

### Fit a linear regression model

In [ ]:
lm = LinearRegression()
lm.fit(X_train, y_train)

### Generate predictions and evaluate the model

We will report two common regression metrics:

- **RMSE**: a measure of typical prediction error size;
- **R-squared**: the proportion of variation explained by the model.



In [ ]:
y_pred = lm.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Test RMSE:", round(rmse, 2))
print("Test R-squared:", round(r2, 3))

### Model coefficients

The coefficients below describe the fitted linear relationship in this training set.
At this point, do not over-interpret them causally.
For now, the main purpose is to see how model output is stored and displayed.

In [ ]:
coef_table = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": lm.coef_
})

coef_table

## 8. Additional practice

1. Create a new working subset that includes `HealthGen` or `Education`.
Then check the missing-value count again.

2. Create another derived variable from an existing numeric variable.
For example, create `LogPulse = np.log(Pulse)` after checking that the variable is positive.

3. Change the list of predictors by removing one variable, such as `LogTotChol`.
Refit the model and compare the new RMSE to the old RMSE.


## 9. Wrap-up

In this lab, you practiced a realistic introductory workflow:

- you reviewed core Python syntax;
- you read a real CSV file;
- you inspected a full public health dataset;
- you selected a working subset of variables;
- you checked and handled missing values;
- you created a derived variable;
- you used that derived variable in later exploration and modeling steps;
- you fit and evaluated a basic regression model.

This sequence is intentionally simple, but it reflects the structure of many real data projects.
In later modules, you will revisit this workflow with more advanced models and evaluation methods.
